# A1 · A classification tree from scratch: Gini, best split, recursion

**Type**: Implement from scratch · **Level**: ●●● challenge · **Time**: about 60 minutes · **extended path**

## Objective

Implement a classification tree with numpy: Gini impurity, search for the **best split** over all the features and thresholds (with cumulative sums, without recounting at every threshold), recursive construction with `max_depth` and `min_samples_leaf`, and prediction.
Check that the root, the structure and the predictions match `DecisionTreeClassifier`.

## Dataset

`load_breast_cancer` (569 nodules, 30 features, 2 classes), stratified 70/30 split: `X_tr, y_tr, X_te, y_te`.

## Educational purpose

Understand what `fit` of a tree does: a greedy algorithm that tries every threshold and picks the largest purity gain. You will see that the cost is dominated by the sorting of the features and that cumulative sums make the search for thresholds linear.

## Task

1. `gini(y)`: Gini impurity of a vector of 0/1 labels (`1 − Σ p_k²`).
2. `best_split(X, y, min_leaf)` → `(gain, feature, threshold)`: for each feature sort the examples, compute with `np.cumsum` the counts of the two classes to the left of each position and the gain $G(\text{parent}) - \frac{n_L G_L + n_R G_R}{n}$. Consider only the positions where the value changes and where both children have at least `min_leaf` examples; the threshold is the midpoint between two consecutive values. If no split has a positive gain return `(0, None, None)`. On ties in gain keep the **first** one found (lowest feature index, lowest threshold).
3. `build(X, y, max_depth, min_leaf)`: recursive tree (dictionaries or tuples). A leaf holds the class counts; stop if the depth is `max_depth`, the node is pure or there is no valid split.
4. `predict(tree, X)`: class for each row (the most frequent one in the leaf). Then compare with `DecisionTreeClassifier(max_depth=4, min_samples_leaf=5, random_state=0)`: save `train_agreement`, `test_agreement` (fraction of equal predictions), `same_root` (feature and threshold of the root) and `same_leaves` (same number of leaves: implement `count_leaves(tree)`).

## Setup

Run this cell without changing it.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

data = load_breast_cancer()
X_tr, X_te, y_tr, y_te = train_test_split(data.data, data.target, test_size=0.3, stratify=data.target, random_state=0)
sk = DecisionTreeClassifier(max_depth=4, min_samples_leaf=5, random_state=0).fit(X_tr, y_tr)

## Your work

In [ ]:
def gini(y):
    ...


def best_split(X, y, min_leaf):
    ...

In [ ]:
def build(X, y, max_depth, min_leaf, depth=0):
    ...


def predict(tree, X):
    ...


def count_leaves(tree):
    ...

In [ ]:
tree = build(X_tr, y_tr, 4, 5)
train_agreement = None
test_agreement = None
same_root = None
same_leaves = None

## Check

Run the cell when you think you are done. If a check fails, the message tells you what is wrong. If it fails a second time, a hint appears under the message.

In [ ]:
with verify(hint='For each feature `j`: `o = np.argsort(X[:, j], kind="stable")`; `xs, ys = X[o, j], y[o]`; `cs = np.cumsum(np.eye(2)[ys], axis=0)` (counts of the two classes on the left); for each `i` from `min_leaf - 1` to `n - min_leaf - 1` with `xs[i] != xs[i + 1]`: `left = cs[i]`, `right = cs[-1] - left`. Gini from counts: `1 - ((c / c.sum()) ** 2).sum()`. Compare with `gain > best + 1e-12` to keep the first one.'):
    check(close(gini(np.array([0, 0, 1, 1])), 0.5) and close(gini(np.array([1, 1, 1])), 0.0) and close(gini(np.array([0, 0, 0, 1])), 0.375), "Gini: 0.5 for half and half, 0 for a pure node, 0.375 for 3 against 1", "1 - sum(p ** 2)")
    g, f, s = best_split(X_tr, y_tr, 5)
    check(f == sk.tree_.feature[0] and abs(s - sk.tree_.threshold[0]) < 1e-3 and g > 0.3, "the best split at the root matches the one of scikit-learn", "cumulative sums on the sorted data; threshold = midpoint")
    check(best_split(np.array([[1.0], [1.0], [1.0], [1.0]]), np.array([0, 1, 0, 1]), 1) == (0, None, None), "no split possible with a constant feature", "consider only the points where the value changes")
    check(train_agreement == 1.0 and test_agreement >= 0.98, "identical predictions on the training set and at least 98% on the test", "scikit-learn compares in float32: a few thresholds may differ")
    check(same_root is True and same_leaves is True and count_leaves(tree) == sk.get_n_leaves(), "same root and same number of leaves", "count_leaves(tree) == sk.get_n_leaves()")
    small = build(X_tr, y_tr, 1, 5)
    check(count_leaves(small) == 2 and (predict(small, X_tr) == DecisionTreeClassifier(max_depth=1, random_state=0).fit(X_tr, y_tr).predict(X_tr)).all(), "with max_depth=1 (a stump): two leaves and the same predictions as scikit-learn", "maximum depth 1")
done()

---
The commented solution is in `solutions/A_from_scratch/A1_classification_tree_from_scratch.ipynb`. Open it after you pass the check, or after you have really tried.